# AIBL Data

---

### package imports and basic functions

---

In [1]:
import os
import gc
import sys
import glob
import shutil
import json
import random
import datetime
import importlib
import itertools
import numpy as np
from scipy import spatial
import scipy.sparse as sparse
import scipy.stats as stats
import pandas as pd
import nibabel as nib
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
from tqdm.auto import tqdm
from urllib.parse import urlparse
import requests
import zipfile
from pathlib import Path
import polars as pl


In [2]:
from spectranorm import snm

In [3]:
class MyNumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        else:
            return super(MyEncoder, self).default(obj)


def ensure_dir(file_name):
    os.makedirs(os.path.dirname(file_name), exist_ok=True)
    return file_name


def list_dirs(path=os.getcwd()):
    files = glob.glob(os.path.join(path, '*'))
    files = [x for x in files if os.path.isdir(x)]
    return files


def file_exists(file_name, path_name=os.getcwd()):
    return os.path.isfile(os.path.join(path_name, file_name))


def write_json(json_obj, file_path):
    with open(file_path, 'w') as outfile:
        json.dump(json_obj, outfile, sort_keys=True, indent=4,
                  cls=MyNumpyEncoder)
    return json_obj


def load_json(file_path):
    with open(file_path, 'r') as infile:
        return json.load(infile)


def write_np(np_obj, file_path):
    with open(file_path, 'wb') as outfile:
        np.save(outfile, np_obj)


## Extracting data

---

In [6]:
# data_info_df = pd.read_csv("/mnt/nas/CSC22/Yeolab/Data/AIBL/users_data/Sina/AIBL.csv")
data_info_df = pd.read_csv("/mnt/nas/CSC22/Yeolab/Data/AIBL/users_data/Sina/AIBL_update_scanpath.csv")  # Prepare by Zhang Chen
data_info_df.shape


(1690, 7)

In [ ]:
data_info_df.head(10)

In [56]:
data_info_df[['SITEID', "Scanner_info"]].value_counts(dropna=False)

SITEID  Scanner_info       
1.0     SIEMENS/TrioTim/3.0    810
NaN     NaN                    415
2.0     SIEMENS/Avanto/1.5     259
        SIEMENS/Verio/3.0      194
NaN     SIEMENS/Avanto/1.5      10
2.0     SIEMENS/TrioTim/3.0      2
Name: count, dtype: int64

In [55]:
data_info_df[['DX']].value_counts(dropna=False)

DX  
CN      1231
MCI      246
AD       204
-4.0       4
7.0        3
NaN        2
Name: count, dtype: int64

In [8]:
num_unique = data_info_df[['SITEID', "Scanner_info"]].drop_duplicates().shape[0]
print(num_unique)

6


In [46]:
num_unique = data_info_df[["RID"]].drop_duplicates().shape[0]
print(num_unique)

862


In [ ]:
list(data_info_df["Scan_path"][:10])

In [ ]:
aibl_valid_subjects_dict = {}

for _, row in tqdm(data_info_df.iterrows()):
    if pd.notna(row["Scan_path"]):
        key = "_".join([str(row["RID"]), str(row["Scan_path"].split("/")[-1])])
        aibl_valid_subjects_dict[key] = {
            "unique_id": key,
            "participant_id": str(row["RID"]),
            "session_id": str(row["Scan_path"].split("/")[-1]),
            "site": f'site:{str(row["SITEID"])}_scanner:{str(row["Scanner_info"])}',
            "sex": row["Sex"],
            "age": row["Age"],
            "scan_path": row["Scan_path"],
        }

len(aibl_valid_subjects_dict), list(aibl_valid_subjects_dict.items())[:1]


In [49]:
items = [
    "lh.white", "rh.white",
    "lh.pial", "rh.pial",
    "lh.thickness", "rh.thickness",
    "lh.orig.nofix", "rh.orig.nofix",
    "lh.sphere.reg", "rh.sphere.reg",
]

def directory_is_valid(path):
    return len([f for f in items if (path / f).exists()]) == len(items)

In [52]:
# Store high-resolution thickness for each individual in a separate file
for idx, subject in enumerate(tqdm(aibl_valid_subjects_dict)):
    sub_dir = f"{idx:02d}"[-2:]

    freesurfer_directory = Path(aibl_valid_subjects_dict[subject]["scan_path"])
    
    thickness_fslr_output = f"/home/sina/storage/Normative_Modeling/data/fs_LR_32k/AIBL/{sub_dir}/{subject}.thickness.fslr.npy"

    if directory_is_valid(freesurfer_directory / "surf") and (not Path(thickness_fslr_output).exists()):    
        # Compute fslr thickness
        transformed_fslr_thickness = snm.utils.nitools.compute_fslr_thickness(freesurfer_directory)
        np.save(
            ensure_dir(thickness_fslr_output),
            transformed_fslr_thickness.astype(np.float32)
        )


  0%|          | 0/1272 [00:00<?, ?it/s]

In [ ]:
%%time
for idx, subject in enumerate(tqdm(aibl_valid_subjects_dict)):
    sub_dir = f"{idx:02d}"[-2:]
    aibl_valid_subjects_dict[subject]["subject_index"] = idx
    thickness_fslr_output = f"/home/sina/storage/Normative_Modeling/data/fs_LR_32k/AIBL/{sub_dir}/{subject}.thickness.fslr.npy"
    if Path(thickness_fslr_output).exists():
        aibl_valid_subjects_dict[subject]["thickness"] = np.load(
            thickness_fslr_output,
        ).mean()
    else:
        aibl_valid_subjects_dict[subject]["thickness"] = np.nan

len(aibl_valid_subjects_dict), list(aibl_valid_subjects_dict.items())[:1]


In [54]:
eno_items = [
    "lh.orig.nofix", "rh.orig.nofix",
]

# Compute Euler Number
for idx, subject in enumerate(tqdm(aibl_valid_subjects_dict)):
    if "euler_no" not in aibl_valid_subjects_dict[subject]:
        sub_dir = f"{idx:02d}"[-2:]
        freesurfer_directory = aibl_valid_subjects_dict[subject]["scan_path"]

        # Compute euler number
        aibl_valid_subjects_dict[subject]["euler_no"] = snm.utils.nitools.compute_total_euler_number(
            Path(freesurfer_directory)
        )


  0%|          | 0/1272 [00:00<?, ?it/s]

In [ ]:
# add site information
for _, row in tqdm(data_info_df.iterrows()):
    if pd.notna(row["Scan_path"]):
        key = "_".join([str(row["RID"]), str(row["Scan_path"].split("/")[-1])])
        aibl_valid_subjects_dict[key]["diagnosis"] = row["DX"] not in ['CN', ]

len(aibl_valid_subjects_dict), list(aibl_valid_subjects_dict.items())[:1]


In [14]:
# Validity checks
for idx, subject in enumerate(tqdm(aibl_valid_subjects_dict)):
    aibl_valid_subjects_dict[subject]["validity_check"] = (
        (aibl_valid_subjects_dict[subject]["diagnosis"] == False)  # Exclude those with a diagnosis
        and
        (aibl_valid_subjects_dict[subject]["thickness"] != np.nan)  # Exclude those missing thickness data
        and
        (aibl_valid_subjects_dict[subject]["euler_no"] != np.nan)  # Exclude those missing Euler number
    )


  0%|          | 0/1272 [00:00<?, ?it/s]

In [15]:
import joblib

dataset = "AIBL"

joblib.dump(aibl_valid_subjects_dict, ensure_dir(f"/home/sina/storage/Normative_Modeling/data/datasets/{dataset}/subjects.joblib"))


['/home/sina/storage/Normative_Modeling/data/datasets/AIBL/subjects.joblib']

In [ ]:
import joblib

# Load the dictionary
valid_subjects_dict = joblib.load(
    f"/home/sina/storage/Normative_Modeling/data/datasets/{dataset}/subjects.joblib"
)

len(valid_subjects_dict), list(valid_subjects_dict.items())[:1]


In [8]:
aibl_valid_subjects_dict = valid_subjects_dict

In [ ]:
final_df = pd.DataFrame({
    'age': [valid_subjects_dict[key]["age"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'thickness': [valid_subjects_dict[key]["thickness"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'sex': [valid_subjects_dict[key]["sex"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'site': [valid_subjects_dict[key]["site"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'subject_ID': [valid_subjects_dict[key]["participant_id"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'euler_no': [valid_subjects_dict[key]["euler_no"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'subject_folder': [valid_subjects_dict[key]["unique_id"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'subject_index': [valid_subjects_dict[key]["subject_index"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
})
final_df['dataset'] = dataset
final_df.head(), final_df.shape


In [21]:
# randomly select only one timepoint per subject (cross-sectional sample)
final_df_subset = final_df.groupby("subject_ID", group_keys=False).sample(n=1, random_state=1234)

# # Only one site:
# final_df_subset.to_parquet(
#     ensure_dir(f'/home/sina/storage/Normative_Modeling/data/datasets/{dataset}/demography.parquet')
# )

# final_df_subset.shape

# Multiple sites:
# Keep only sites with at least 15 subjects
subjects_per_site = final_df_subset.groupby("site")["subject_ID"].nunique()
valid_sites = subjects_per_site[subjects_per_site >= 15].index

final_df_subset[final_df_subset["site"].isin(valid_sites)].to_parquet(
    ensure_dir(f'/home/sina/storage/Normative_Modeling/data/datasets/{dataset}/demography.parquet')
)

final_df_subset[final_df_subset["site"].isin(valid_sites)].shape


(494, 9)

# ✅ Finished!
